# P9-IMPL-NOAA-WEATHER-INGEST — NWS Weather Alert Ingestion Pipeline

**Author:** @data-engineer | **Date:** 2026-10-21  
**Schema:** `cdm_tmforum.provable_resilience_intelligence`  
**Source:** [api.weather.gov/alerts](https://api.weather.gov/alerts)  
**Refresh:** 15-minute cadence (scheduled)  
**SID Domain:** Common → GeographicPlace → HazardEvent  

## Pipeline Architecture
```
NWS API (GeoJSON) → Fetch+Parse → GeoJSON→WKT Transform → stg_noaa_weather_alert (Bronze)
                                                         ↓ MERGE
                                                    geographic_hazard_event (Gold)
```

## Lakelink Fiber — Provable Resilience Intelligence

In [0]:
%sql
-- Create the stg_noaa_weather_alert staging table (idempotent)
CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_noaa_weather_alert (
  alert_id              STRING        NOT NULL  COMMENT 'NWS alert URI (e.g. urn:oid:2.49.0.1.840.0...). Globally unique.',
  event_type            STRING        NOT NULL  COMMENT 'NWS event name: Tornado Warning, Severe Thunderstorm Warning, Winter Storm Warning, Flood Warning, etc.',
  severity              STRING        NOT NULL  COMMENT 'CAP severity: Extreme | Severe | Moderate | Minor | Unknown',
  certainty             STRING        NOT NULL  COMMENT 'CAP certainty: Observed | Likely | Possible | Unlikely | Unknown',
  urgency               STRING        NOT NULL  COMMENT 'CAP urgency: Immediate | Expected | Future | Past | Unknown',
  response_type         STRING                  COMMENT 'CAP response: Shelter | Evacuate | Prepare | Execute | Avoid | Monitor | None',
  onset                 TIMESTAMP     NOT NULL  COMMENT 'When the alert hazard begins (UTC)',
  expires               TIMESTAMP               COMMENT 'When the alert expires (UTC). NULL if open-ended.',
  effective             TIMESTAMP               COMMENT 'When the alert information becomes effective (UTC)',
  sent                  TIMESTAMP     NOT NULL  COMMENT 'When the alert was issued by NWS (UTC)',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POLYGON or MULTIPOLYGON from GeoJSON geometry. WGS84.',
  centroid_lat          DOUBLE                  COMMENT 'Centroid latitude computed from geom_wkt for proximity queries',
  centroid_lon          DOUBLE                  COMMENT 'Centroid longitude computed from geom_wkt for proximity queries',
  affected_zones        STRING                  COMMENT 'Comma-separated NWS zone codes (e.g. TXZ104,TXZ105)',
  area_desc             STRING                  COMMENT 'Human-readable area description from NWS',
  headline              STRING                  COMMENT 'NWS headline text for the alert',
  description           STRING                  COMMENT 'Full NWS alert description text',
  instruction           STRING                  COMMENT 'NWS recommended protective actions',
  sender_name           STRING                  COMMENT 'Issuing NWS Weather Forecast Office name',
  source_url            STRING        NOT NULL  COMMENT 'Full API URL that returned this alert',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)',
  is_active             BOOLEAN       NOT NULL  COMMENT 'TRUE if alert is currently active (expires > NOW())'
)
COMMENT 'Staging: NWS severe weather alerts from api.weather.gov. One row per alert. Refreshed every 15 min. Source for geographic_hazard_event severe_weather enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT15M',
  'source.api' = 'api.weather.gov/alerts',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
)

In [0]:
"""Fetch active NWS weather alerts and transform GeoJSON→WKT for Lakelink Fiber PRI."""
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, BooleanType
)

# --- API Configuration ---
NOAA_ALERTS_URL = "https://api.weather.gov/alerts/active"
HEADERS = {
    "User-Agent": "(Lakelink Fiber PRI Pipeline, pri-pipeline@lakelink-fiber.com)",
    "Accept": "application/geo+json"
}
SCHEMA = "cdm_tmforum.provable_resilience_intelligence"

# --- GeoJSON → WKT Conversion ---
def coords_to_wkt_polygon(coords):
    """Convert GeoJSON Polygon coordinates to WKT POLYGON."""
    rings = []
    for ring in coords:
        pts = ", ".join(f"{lon} {lat}" for lon, lat in ring)
        rings.append(f"({pts})")
    return f"POLYGON({', '.join(rings)})"

def coords_to_wkt_multipolygon(coords):
    """Convert GeoJSON MultiPolygon coordinates to WKT MULTIPOLYGON."""
    polygons = []
    for poly_coords in coords:
        rings = []
        for ring in poly_coords:
            pts = ", ".join(f"{lon} {lat}" for lon, lat in ring)
            rings.append(f"({pts})")
        polygons.append(f"({', '.join(rings)})")
    return f"MULTIPOLYGON({', '.join(polygons)})"

def geojson_to_wkt(geometry):
    """Convert a GeoJSON geometry dict to WKT string."""
    if geometry is None:
        return None
    geo_type = geometry.get("type", "").lower()
    coords = geometry.get("coordinates", [])
    if geo_type == "polygon":
        return coords_to_wkt_polygon(coords)
    elif geo_type == "multipolygon":
        return coords_to_wkt_multipolygon(coords)
    elif geo_type == "point":
        return f"POINT({coords[0]} {coords[1]})"
    return None

def compute_centroid(geometry):
    """Compute approximate centroid from GeoJSON geometry."""
    if geometry is None:
        return (None, None)
    geo_type = geometry.get("type", "").lower()
    coords = geometry.get("coordinates", [])
    if geo_type == "point":
        return (coords[1], coords[0])
    elif geo_type == "polygon":
        ring = coords[0] if coords else []
        if not ring:
            return (None, None)
        lats = [pt[1] for pt in ring]
        lons = [pt[0] for pt in ring]
        return (sum(lats) / len(lats), sum(lons) / len(lons))
    elif geo_type == "multipolygon":
        all_pts = [pt for poly in coords for ring in poly for pt in ring]
        if not all_pts:
            return (None, None)
        lats = [pt[1] for pt in all_pts]
        lons = [pt[0] for pt in all_pts]
        return (sum(lats) / len(lats), sum(lons) / len(lons))
    return (None, None)

def parse_timestamp(ts_str):
    """Parse ISO 8601 timestamp from NWS API."""
    if not ts_str:
        return None
    try:
        # Handle various ISO formats the NWS API returns
        ts_str = ts_str.replace("Z", "+00:00")
        return datetime.fromisoformat(ts_str).astimezone(timezone.utc).replace(tzinfo=None)
    except Exception:
        return None

# --- Fetch alerts ---
print(f"Fetching active alerts from {NOAA_ALERTS_URL}...")
response = requests.get(NOAA_ALERTS_URL, headers=HEADERS, timeout=30)
response.raise_for_status()
data = response.json()
features = data.get("features", [])
print(f"Received {len(features)} active alerts from NWS")

# --- Transform ---
now = datetime.now(timezone.utc).replace(tzinfo=None)
rows = []
skipped = 0
for feat in features:
    props = feat.get("properties", {})
    geom = feat.get("geometry")
    alert_id = props.get("id")
    event_type = props.get("event")
    severity = props.get("severity")
    certainty = props.get("certainty")
    urgency = props.get("urgency")
    sent = parse_timestamp(props.get("sent"))
    onset = parse_timestamp(props.get("onset")) or sent  # fallback to sent
    
    # Skip rows missing required fields
    if not all([alert_id, event_type, severity, certainty, urgency, sent]):
        skipped += 1
        continue
    
    wkt = geojson_to_wkt(geom)
    centroid_lat, centroid_lon = compute_centroid(geom)
    
    # Fallback for null-geometry alerts (99%+ of NWS alerts use zone references)
    # Extract state code from UGC geocodes and use state centroid as POINT geometry
    if wkt is None:
        STATE_CENTROIDS = {
            'AL': (32.8,-86.8), 'AK': (64.2,-152.5), 'AZ': (34.3,-111.7), 'AR': (34.8,-92.2),
            'CA': (37.2,-119.5), 'CO': (39.0,-105.5), 'CT': (41.6,-72.7), 'DE': (39.0,-75.5),
            'FL': (28.6,-82.4), 'GA': (32.7,-83.5), 'HI': (20.5,-156.4), 'ID': (44.4,-114.6),
            'IL': (40.0,-89.2), 'IN': (39.9,-86.3), 'IA': (42.0,-93.5), 'KS': (38.5,-98.4),
            'KY': (37.8,-85.3), 'LA': (31.0,-92.0), 'ME': (45.4,-69.2), 'MD': (39.0,-76.8),
            'MA': (42.3,-71.8), 'MI': (44.3,-84.6), 'MN': (46.3,-94.3), 'MS': (32.7,-89.7),
            'MO': (38.4,-92.5), 'MT': (47.1,-109.6), 'NE': (41.5,-99.8), 'NV': (39.3,-116.6),
            'NH': (43.7,-71.6), 'NJ': (40.1,-74.7), 'NM': (34.4,-106.1), 'NY': (42.9,-75.5),
            'NC': (35.5,-79.4), 'ND': (47.4,-100.5), 'OH': (40.4,-82.8), 'OK': (35.6,-97.5),
            'OR': (44.0,-120.5), 'PA': (40.9,-77.8), 'RI': (41.7,-71.5), 'SC': (33.9,-80.9),
            'SD': (44.4,-100.2), 'TN': (35.9,-86.4), 'TX': (31.5,-99.3), 'UT': (39.3,-111.7),
            'VT': (44.1,-72.6), 'VA': (37.5,-78.9), 'WA': (47.4,-120.7), 'WV': (38.6,-80.6),
            'WI': (44.6,-89.8), 'WY': (43.0,-107.6), 'DC': (38.9,-77.0),
            'PR': (18.2,-66.5), 'VI': (18.3,-64.9), 'GU': (13.4, 144.8), 'AS': (-14.3,-170.7),
            'MP': (15.2, 145.7), 'PW': (7.5, 134.6), 'MH': (7.1, 171.2),
            'AM': (18.2,-66.5), 'GM': (28.0,-90.0), 'PH': (18.2,-66.5), 'LC': (18.2,-66.5),
            'LM': (44.0,-86.0), 'LH': (45.0,-83.5), 'LE': (42.2,-81.0), 'LO': (43.5,-77.0),
            'LS': (47.5,-88.0), 'SL': (43.0,-79.0)
        }
        ugc_codes = props.get('geocode', {}).get('UGC', [])
        zone_lats, zone_lons = [], []
        for ugc in ugc_codes:
            state_code = ugc[:2]
            if state_code in STATE_CENTROIDS:
                lat, lon = STATE_CENTROIDS[state_code]
                zone_lats.append(lat)
                zone_lons.append(lon)
        if zone_lats:
            centroid_lat = sum(zone_lats) / len(zone_lats)
            centroid_lon = sum(zone_lons) / len(zone_lons)
            wkt = f"POINT({centroid_lon} {centroid_lat})"
        else:
            skipped += 1
            continue
    expires = parse_timestamp(props.get("expires"))
    is_active = expires is None or expires > now
    
    affected_zones_list = props.get("affectedZones", [])
    zone_codes = ",".join(z.split("/")[-1] for z in affected_zones_list) if affected_zones_list else None
    
    rows.append(Row(
        alert_id=alert_id,
        event_type=event_type,
        severity=severity,
        certainty=certainty,
        urgency=urgency,
        response_type=",".join(props.get("response", [])) if props.get("response") else props.get("response", None),
        onset=onset,
        expires=expires,
        effective=parse_timestamp(props.get("effective")),
        sent=sent,
        geom_wkt=wkt,
        centroid_lat=centroid_lat,
        centroid_lon=centroid_lon,
        affected_zones=zone_codes,
        area_desc=props.get("areaDesc"),
        headline=props.get("headline"),
        description=props.get("description"),
        instruction=props.get("instruction"),
        sender_name=props.get("senderName"),
        source_url=f"https://api.weather.gov/alerts/{alert_id}",
        ingested_at=now,
        raw_json=json.dumps(feat),
        is_active=is_active
    ))

print(f"Transformed {len(rows)} alerts ({skipped} skipped for missing geometry/required fields)")

In [0]:
# Upsert transformed alerts into staging table
if len(rows) == 0:
    print("⚠️ No alerts to upsert. NWS may have returned empty results (common during low-activity periods).")
else:
    schema = StructType([
        StructField("alert_id", StringType(), False),
        StructField("event_type", StringType(), False),
        StructField("severity", StringType(), False),
        StructField("certainty", StringType(), False),
        StructField("urgency", StringType(), False),
        StructField("response_type", StringType(), True),
        StructField("onset", TimestampType(), False),
        StructField("expires", TimestampType(), True),
        StructField("effective", TimestampType(), True),
        StructField("sent", TimestampType(), False),
        StructField("geom_wkt", StringType(), False),
        StructField("centroid_lat", DoubleType(), True),
        StructField("centroid_lon", DoubleType(), True),
        StructField("affected_zones", StringType(), True),
        StructField("area_desc", StringType(), True),
        StructField("headline", StringType(), True),
        StructField("description", StringType(), True),
        StructField("instruction", StringType(), True),
        StructField("sender_name", StringType(), True),
        StructField("source_url", StringType(), False),
        StructField("ingested_at", TimestampType(), False),
        StructField("raw_json", StringType(), True),
        StructField("is_active", BooleanType(), False),
    ])
    
    df_alerts = spark.createDataFrame(rows, schema=schema)
    df_alerts.createOrReplaceTempView("new_alerts")
    
    # MERGE upsert: update existing alerts, insert new ones
    result = spark.sql(f"""
        MERGE INTO {SCHEMA}.stg_noaa_weather_alert AS target
        USING new_alerts AS source
        ON target.alert_id = source.alert_id
        WHEN MATCHED THEN UPDATE SET
            target.severity = source.severity,
            target.certainty = source.certainty,
            target.urgency = source.urgency,
            target.response_type = source.response_type,
            target.expires = source.expires,
            target.geom_wkt = source.geom_wkt,
            target.centroid_lat = source.centroid_lat,
            target.centroid_lon = source.centroid_lon,
            target.headline = source.headline,
            target.description = source.description,
            target.instruction = source.instruction,
            target.ingested_at = source.ingested_at,
            target.raw_json = source.raw_json,
            target.is_active = source.is_active
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    # Report metrics
    stg_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {SCHEMA}.stg_noaa_weather_alert").first()[0]
    active_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {SCHEMA}.stg_noaa_weather_alert WHERE is_active = TRUE").first()[0]
    print(f"✅ Staging upsert complete: {len(rows)} alerts processed")
    print(f"   Total staging rows: {stg_count} | Active: {active_count}")

In [0]:
# Enrich geographic_hazard_event with new weather alerts
# Map NWS severity to PRI severity and generate deterministic hazard_id

result = spark.sql(f"""
    MERGE INTO {SCHEMA}.geographic_hazard_event AS gold
    USING (
        SELECT
            CONCAT('HAZ-NWS-', SUBSTR(MD5(alert_id), 1, 8)) AS hazard_id,
            'severe_weather' AS hazard_type,
            CASE severity
                WHEN 'Extreme' THEN 'critical'
                WHEN 'Severe' THEN 'high'
                WHEN 'Moderate' THEN 'medium'
                WHEN 'Minor' THEN 'low'
                ELSE 'low'
            END AS severity,
            centroid_lat AS latitude,
            centroid_lon AS longitude,
            -- Estimate impact radius from event type
            CASE 
                WHEN event_type LIKE '%Tornado%' THEN 25.0
                WHEN event_type LIKE '%Hurricane%' THEN 200.0
                WHEN event_type LIKE '%Severe Thunderstorm%' THEN 50.0
                WHEN event_type LIKE '%Winter Storm%' THEN 100.0
                WHEN event_type LIKE '%Flood%' THEN 30.0
                WHEN event_type LIKE '%Ice Storm%' THEN 75.0
                ELSE 40.0
            END AS radius_km,
            geom_wkt,
            onset AS event_start,
            expires AS event_end,
            'nws' AS source,
            CONCAT(event_type, ': ', COALESCE(headline, area_desc, 'NWS alert')) AS description,
            ingested_at AS created_at
        FROM {SCHEMA}.stg_noaa_weather_alert
        WHERE is_active = TRUE
    ) AS stg
    ON gold.hazard_id = stg.hazard_id
    WHEN MATCHED THEN UPDATE SET
        gold.severity = stg.severity,
        gold.event_end = stg.event_end,
        gold.description = stg.description,
        gold.geom_wkt = stg.geom_wkt
    WHEN NOT MATCHED THEN INSERT *
""")

gold_count = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event").first()[0]
nws_count = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'nws'").first()[0]
print(f"✅ Gold MERGE complete. geographic_hazard_event: {gold_count:,} total | {nws_count:,} NWS alerts")

In [0]:
# Mark expired alerts as inactive in staging
result = spark.sql(f"""
    UPDATE {SCHEMA}.stg_noaa_weather_alert
    SET is_active = FALSE
    WHERE expires < CURRENT_TIMESTAMP() AND is_active = TRUE
""")

# Update event_end in gold for expired alerts  
result = spark.sql(f"""
    MERGE INTO {SCHEMA}.geographic_hazard_event AS gold
    USING (
        SELECT 
            CONCAT('HAZ-NWS-', SUBSTR(MD5(alert_id), 1, 8)) AS hazard_id,
            expires AS event_end
        FROM {SCHEMA}.stg_noaa_weather_alert
        WHERE is_active = FALSE AND expires IS NOT NULL
    ) AS expired
    ON gold.hazard_id = expired.hazard_id AND gold.event_end IS NULL
    WHEN MATCHED THEN UPDATE SET gold.event_end = expired.event_end
""")

print("✅ Expired alerts deactivated and event_end timestamps backfilled")

In [0]:
# Final pipeline run summary
from datetime import datetime, timezone

stg_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_noaa_weather_alert").first()[0]
stg_active = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_noaa_weather_alert WHERE is_active").first()[0]
gold_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event").first()[0]
gold_nws = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'nws'").first()[0]

severity_dist = spark.sql(f"""
    SELECT severity, COUNT(*) as cnt 
    FROM {SCHEMA}.stg_noaa_weather_alert 
    WHERE is_active GROUP BY severity ORDER BY cnt DESC
""").collect()

print("=" * 60)
print("NOAA WEATHER ALERT INGESTION — RUN COMPLETE")
print(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60)
print(f"Staging (stg_noaa_weather_alert): {stg_total:,} total | {stg_active:,} active")
print(f"Gold (geographic_hazard_event):   {gold_total:,} total | {gold_nws:,} from NWS")
if severity_dist:
    print(f"Active alert severity: {', '.join(f'{r.severity}={r.cnt}' for r in severity_dist)}")
print("=" * 60)